# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose a **Random Forest Regressor**.

The lane is a ranking problem: the useful output is an ordered queue of content items for human review. Instead of predicting a class, the model predicts a future-performance proxy (`future_clicks`) and the predictions are used to rank items.

Random Forest is a reasonable first model because it can capture nonlinear relationships between search demand, clicks, CTR, position and analytics signals without requiring a large amount of feature engineering. It is also interpretable enough for a first comparison using permutation importance.

The model is deliberately compared against the simple Week-4 rule. More complexity is only useful if it improves the decision metric on unseen clients.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

from huggingface_hub import HfApi, hf_hub_download
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit

# Authentication check
whoami = HfApi().whoami()
print("Logged in as:", whoami["name"])

# Download the two months needed for an honest past -> future experiment.
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
)

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
)

print("March:", march_path)
print("April:", april_path)

con = duckdb.connect()


Logged in as: Parasiticwire
March: C:\Users\zain\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet
April: C:\Users\zain\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-04\data_0.parquet


c:\Users\zain\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zain\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 2. Split design

The split is **grouped by client**.

All content belonging to a client stays entirely in either train or validation. This prevents the model from learning client-specific patterns from some content and then being tested on other content from the same client.

The March data is the feature window. April clicks are the future outcome proxy. Therefore April information is never used as a feature.

The same held-out validation clients are used for both the model and the Week-4 baseline.

In [4]:
# Build one row per client-content pair.
# March = features available at the decision moment.
# April = future outcome used only for evaluation.

feature_query = f'''
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(ga4_pageviews) AS pageviews,
        SUM(ga4_sessions) AS sessions,
        SUM(sessions_organic) AS sessions_organic,
        SUM(sessions_direct) AS sessions_direct,
        SUM(sessions_referral) AS sessions_referral,
        SUM(sessions_social) AS sessions_social,
        SUM(sessions_paid) AS sessions_paid,
        SUM(sessions_ai) AS sessions_ai,
        SUM(scroll_events) AS scroll_events,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_clicks
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
)
SELECT
    m.*,
    COALESCE(a.future_clicks, 0) AS future_clicks
FROM march m
LEFT JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
WHERE m.impressions > 0
'''

data = con.sql(feature_query).df()

data["ctr_pct"] = 100.0 * data["clicks"] / data["impressions"].replace(0, np.nan)
data["pageviews_per_session"] = data["pageviews"] / data["sessions"].replace(0, np.nan)

data = data.replace([np.inf, -np.inf], np.nan)
data = data.fillna(0)

print("Rows:", len(data))
print("Clients:", data["client_hash_id"].nunique())
print("Future clicks:", data["future_clicks"].sum())
data.head()


Rows: 176738
Clients: 47
Future clicks: 791346.0


,client_hash_id,content_hash_id,impressions,clicks,pageviews,sessions,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,scroll_events,avg_position,future_clicks,ctr_pct,pageviews_per_session
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,345.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.879856,0.0,0.289855,0.0
1,client_62f4a7e64f5e0096,content_cd3ce62c4e1d7b5a,162.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.288664,0.0,0.000000,0.0
2,client_62f4a7e64f5e0096,content_b1973b64637339e2,826.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.342134,1.0,0.726392,0.0
3,client_62f4a7e64f5e0096,content_20df056093d0600e,1277.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.812888,6.0,0.548160,0.0
4,client_62f4a7e64f5e0096,content_1b17c92ca7ad4d40,5768.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.155372,3.0,0.260055,0.0


In [5]:
# Grouped train/validation split.
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, valid_idx = next(gss.split(data, groups=groups))

train = data.iloc[train_idx].copy()
valid = data.iloc[valid_idx].copy()

print("Train rows:", len(train))
print("Validation rows:", len(valid))
print("Train clients:", train["client_hash_id"].nunique())
print("Validation clients:", valid["client_hash_id"].nunique())
print("Client overlap:", len(set(train.client_hash_id) & set(valid.client_hash_id)))


Train rows: 138310
Validation rows: 38428
Train clients: 37
Validation clients: 10
Client overlap: 0


## 3. Train + compare vs my baseline

### Target

The model predicts `future_clicks`: the number of GSC clicks observed in the following month.

This is a **proxy for future content performance**, not a guarantee that a recommendation will cause more clicks.

### Features

Only March information is used:

- `impressions`
- `clicks`
- `ctr_pct`
- `avg_position`
- `pageviews`
- `sessions`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `scroll_events`
- `pageviews_per_session`

The future label is never included as a feature.

### Ranking metric

We use **NDCG@10**, calculated separately within each client and then averaged. This asks whether the top of the ranked queue contains items with high future click outcomes.

The Week-4 rule and Random Forest are evaluated on exactly the same validation rows and exactly the same metric.

In [6]:
FEATURES = [
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
    "pageviews",
    "sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
    "pageviews_per_session",
]

X_train = train[FEATURES]
y_train = train["future_clicks"]

X_valid = valid[FEATURES]
y_valid = valid["future_clicks"]

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

valid["model_score"] = model.predict(X_valid)

print("Model trained.")
print("Validation MAE:", round(mean_absolute_error(y_valid, valid["model_score"]), 4))


Model trained.
Validation MAE: 3.2461


In [7]:
# Recreate the Week-4 transparent baseline on the SAME March feature rows.

valid["baseline_score"] = np.select(
    [
        (valid["avg_position"] > 3) &
        (valid["avg_position"] <= 10) &
        (valid["ctr_pct"] < 2),

        (valid["avg_position"] > 10) &
        (valid["impressions"] >= 50),

        valid["impressions"] >= 50,
    ],
    [3.0, 2.0, 1.0],
    default=0.0,
)

valid["baseline_reason"] = np.select(
    [
        (valid["avg_position"] > 3) &
        (valid["avg_position"] <= 10) &
        (valid["ctr_pct"] < 2),

        (valid["avg_position"] > 10) &
        (valid["impressions"] >= 50),

        valid["impressions"] >= 50,
    ],
    [
        "LOW_CTR_MID_POSITION",
        "HIGH_VOLUME_LOW_POSITION",
        "MEANINGFUL_SEARCH_VOLUME",
    ],
    default="NO_STRONG_SIGNAL",
)

valid[["client_hash_id","content_hash_id","future_clicks","baseline_score","model_score"]].head()


,client_hash_id,content_hash_id,future_clicks,baseline_score,model_score
44,client_9958f0a7ae1df715,content_b14b1b3afddc0287,0.0,2.0,0.058497
45,client_9958f0a7ae1df715,content_94798aa612b0e422,0.0,2.0,1.058752
46,client_9958f0a7ae1df715,content_41b7a92fa1621773,0.0,3.0,1.287161
47,client_9958f0a7ae1df715,content_6486ebe4abb9e7fb,2.0,2.0,0.621932
48,client_9958f0a7ae1df715,content_e5e77cb12c582d30,0.0,2.0,0.156493


In [8]:
def dcg(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(values) + 2))
    return np.sum((2 ** values - 1) / discounts)

def ndcg_at_k(group, score_col, k=10):
    ordered = group.sort_values(score_col, ascending=False)
    actual = ordered["future_clicks"].to_numpy()[:k]

    ideal = np.sort(group["future_clicks"].to_numpy())[::-1][:k]

    ideal_dcg = dcg(ideal)
    if ideal_dcg == 0:
        return np.nan

    return dcg(actual) / ideal_dcg

def mean_client_ndcg(frame, score_col, k=10):
    scores = []
    for _, g in frame.groupby("client_hash_id"):
        value = ndcg_at_k(g, score_col, k)
        if not np.isnan(value):
            scores.append(value)
    return float(np.mean(scores)) if scores else np.nan

baseline_ndcg = mean_client_ndcg(valid, "baseline_score", k=10)
model_ndcg = mean_client_ndcg(valid, "model_score", k=10)

comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "NDCG@10": [baseline_ndcg, model_ndcg],
})

comparison["delta_vs_baseline"] = comparison["NDCG@10"] - baseline_ndcg
comparison


C:\Users\zain\AppData\Local\Temp\ipykernel_1256\320548579.py:6: RuntimeWarning: overflow encountered in power
  return np.sum((2 ** values - 1) / discounts)
C:\Users\zain\AppData\Local\Temp\ipykernel_1256\320548579.py:18: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(actual) / ideal_dcg


,method,NDCG@10,delta_vs_baseline
0,Week-4 baseline,0.060638,0.000000
1,Random Forest,0.850780,0.790142


### Model-vs-baseline interpretation

Use the table above to write the result in careful language.

- If Random Forest has higher NDCG@10: **"On this held-out client split, the model ranked future-click outcomes better than the baseline according to NDCG@10."**
- If it is lower: **"On this held-out client split, the baseline ranked future-click outcomes better than the model according to NDCG@10."**
- If they are close: **"The model did not show a meaningful improvement over the simple baseline on this split."**

Do not claim that the model will improve traffic in production. This experiment only measures ranking performance against the selected future-click proxy.


## 4. Errors and interpretation

First inspect where the model disagrees with the future outcome and which features it relies on.

A useful error is not just "the prediction was numerically wrong". For this lane, a ranking error means an item was placed high in the queue even though its future outcome was weak, or an item with a strong future outcome was pushed too low.

In [9]:
# Largest absolute prediction errors
valid["abs_error"] = (valid["future_clicks"] - valid["model_score"]).abs()

error_rows = valid.sort_values("abs_error", ascending=False)[[
    "client_hash_id",
    "content_hash_id",
    "future_clicks",
    "model_score",
    "baseline_score",
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
]]

error_rows.head(10)


,client_hash_id,content_hash_id,future_clicks,model_score,baseline_score,impressions,clicks,ctr_pct,avg_position
57215,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,2050.0,1167.495107,1.0,28337.0,803.0,2.833751,3.043625
37316,client_0fa64a184f18a4a0,content_2bed2c9ce7808050,924.0,232.044882,1.0,13314.0,248.0,1.862701,1.574666
11822,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,1845.0,1191.890702,3.0,154358.0,2506.0,1.623499,3.019798
84382,client_0fa64a184f18a4a0,content_2db2a9dcb3b62a3a,1094.0,459.892578,1.0,23876.0,462.0,1.934997,2.554614
97571,client_73cda7b4e4f265ea,content_43c7cd15278a7938,581.0,1.778675,1.0,319.0,2.0,0.626959,1.815268
155722,client_73cda7b4e4f265ea,content_987d251ee617d9c6,674.0,1177.695705,1.0,152806.0,940.0,0.615159,2.823429
101764,client_73cda7b4e4f265ea,content_65f0084145088393,805.0,306.906801,3.0,65268.0,376.0,0.576086,4.407644
46223,client_0fa64a184f18a4a0,content_a4f6ae139e17a390,457.0,7.858784,1.0,700.0,8.0,1.142857,2.533855
68821,client_73cda7b4e4f265ea,content_85703b835ab9e744,736.0,1181.867959,1.0,120868.0,1091.0,0.902638,2.728857
17703,client_0fa64a184f18a4a0,content_d6c8477d8d3825dd,390.0,5.806932,1.0,1404.0,6.0,0.427350,1.274841


In [10]:
# Permutation importance on the held-out validation set.
# This measures how much the model's predictive score changes when one feature
# is shuffled. It is not causal importance.

perm = permutation_importance(
    model,
    X_valid,
    y_valid,
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance.head(10)


,feature,importance_mean,importance_std
1,clicks,1.415035e+00,0.006704
13,pageviews_per_session,4.353087e-03,0.000967
0,impressions,2.686481e-03,0.000224
3,avg_position,2.043556e-03,0.000631
7,sessions_direct,1.897161e-03,0.000132
2,ctr_pct,9.187936e-04,0.000430
6,sessions_organic,5.001657e-04,0.000080
10,sessions_paid,3.492121e-04,0.000127
4,pageviews,3.017793e-04,0.000032
11,sessions_ai,7.884915e-07,0.000010


### Error interpretation

After running the cells above, write 3–5 sentences:

1. **What the model got wrong:** describe a visible pattern in the largest errors.
2. **What the baseline got right/wrong:** compare the top-ranked items rather than only looking at the score.
3. **What the model leaned on:** use the permutation-importance table.
4. **One limitation:** name a reason the April future-click proxy may not fully represent the real content-review decision.
5. **Decision:** say whether the model is worth carrying forward, based on the held-out comparison—not on complexity.


In [11]:
# Compare the top 10 model and baseline queues on the same validation set.
top10_comparison = pd.concat([
    valid.sort_values("model_score", ascending=False).head(10).assign(queue="Model"),
    valid.sort_values("baseline_score", ascending=False).head(10).assign(queue="Baseline")
])[[
    "queue",
    "client_hash_id",
    "content_hash_id",
    "future_clicks",
    "model_score",
    "baseline_score",
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position"
]]

top10_comparison


,queue,client_hash_id,content_hash_id,future_clicks,model_score,baseline_score,impressions,clicks,ctr_pct,avg_position
56233,Model,client_73cda7b4e4f265ea,content_17494d099b0a537e,884.0,1196.540813,1.0,87928.0,935.0,1.063370,2.797686
11822,Model,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,1845.0,1191.890702,3.0,154358.0,2506.0,1.623499,3.019798
153005,Model,client_73cda7b4e4f265ea,content_6c4da03688ca8351,1173.0,1187.924565,3.0,76184.0,938.0,1.231230,4.019295
68821,Model,client_73cda7b4e4f265ea,content_85703b835ab9e744,736.0,1181.867959,1.0,120868.0,1091.0,0.902638,2.728857
155722,Model,client_73cda7b4e4f265ea,content_987d251ee617d9c6,674.0,1177.695705,1.0,152806.0,940.0,0.615159,2.823429
57215,Model,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,2050.0,1167.495107,1.0,28337.0,803.0,2.833751,3.043625
66129,Model,client_73cda7b4e4f265ea,content_57c3b90b328b406e,751.0,925.997774,1.0,82298.0,735.0,0.893096,2.708173
24288,Model,client_73cda7b4e4f265ea,content_5267d90f451c6edc,671.0,836.476849,1.0,51920.0,707.0,1.361710,1.637758
150547,Model,client_73cda7b4e4f265ea,content_b2cb08ff59fcce78,466.0,571.118082,3.0,106025.0,663.0,0.625324,3.235548
2154,Model,client_73cda7b4e4f265ea,content_57768353f230d65d,434.0,549.881688,3.0,114479.0,581.0,0.507517,3.326894


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.